In [15]:
import pandas as pd

# Load the SMS spam dataset, separate columns using tab, and assign column names
messages = pd.read_csv(
    '../data/sms_spam/SMSSpamCollection.txt',
    sep='\t',
    names=["label", "message"]
)

In [16]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


#### Data Cleaning And Preprocessing

In [17]:
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rmahf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
porter_stemmer = PorterStemmer()

In [19]:
corpus = []
stop_words = set(stopwords.words('english'))

for i in range(len(messages)):
    review = messages['message'][i] # Get the current SMS message
    review = re.sub('[^a-zA-Z]', ' ', review) # Remove non-alphabetic characters
    review = review.lower()
    review = review.split()
    review = [porter_stemmer.stem(word) for word in review if not word  in stop_words]
    review = ' '.join(review) # Join the words back into a sentence
    corpus.append(review) # Add the processed message to the corpus

In [ ]:
# Independent feature 
corpus

['go jurong point crazi avail bugi n great world la e buffet cine got amor wat',
 'ok lar joke wif u oni',
 'free entri wkli comp win fa cup final tkt st may text fa receiv entri question std txt rate c appli',
 'u dun say earli hor u c alreadi say',
 'nah think goe usf live around though',
 'freemsg hey darl week word back like fun still tb ok xxx std chg send rcv',
 'even brother like speak treat like aid patent',
 'per request mell mell oru minnaminungint nurungu vettam set callertun caller press copi friend callertun',
 'winner valu network custom select receivea prize reward claim call claim code kl valid hour',
 'mobil month u r entitl updat latest colour mobil camera free call mobil updat co free',
 'gonna home soon want talk stuff anymor tonight k cri enough today',
 'six chanc win cash pound txt csh send cost p day day tsandc appli repli hl info',
 'urgent week free membership prize jackpot txt word claim c www dbuk net lccltd pobox ldnw rw',
 'search right word thank breather

In [22]:
# Output feature
y = messages['label'].map({'ham': 0, 'spam': 1}).values
y


array([0, 0, 1, ..., 0, 0, 0], shape=(5572,))


#### BOW, TF-IDF & Machine Learning Pipeline

- **Preprocessing and Cleaning** — lowercase text, remove punctuation/numbers, tokenize, remove stopwords, apply stemming or lemmatization
- **Train-Test Split** — split cleaned data into train and test sets **before** vectorization
- **BOW/TF-IDF (Preventing Data Leakage)** — fit the vectorizer only on training data, then use that same fitted vectorizer to transform both train and test data
- **Why this matters** — fitting on the full dataset (train + test) leaks test information into training, giving falsely inflated accuracy
- **Train the Model** — feed vectorized training data into an ML algorithm (Naive Bayes, Logistic Regression, SVM, etc.) and evaluate on vectorized test data

```text
Text → Cleaning → Train/Test Split → BoW / TF-IDF → ML Model → Prediction
```


#### Create Bag Of Words

##### Train Test Split

In [23]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(corpus,y,test_size=0.20)


In [24]:
len(X_train), len(y_train)

(4457, 4457)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

# Create the Bag OF Words model
count_vector = CountVectorizer(max_features=2500, binary = False, ngram_range=(1,2))

In [26]:
X_train_bow = count_vector.fit_transform(X_train).toarray()
X_test_bow = count_vector.transform(X_test).toarray()

In [27]:
import numpy as np

np.set_printoptions(
    precision=3,       # Show numbers with 3 decimal places
    suppress=True,     # Avoid scientific notation for small numbers
    edgeitems=30,      # Show up to 30 items at the beginning and end
    linewidth=100000   # Allow a very wide output line
)

X_train_bow

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0

In [28]:
count_vector.vocabulary_

{'hi': np.int64(934),
 'pleas': np.int64(1603),
 'get': np.int64(778),
 'lt': np.int64(1249),
 'gt': np.int64(869),
 'dollar': np.int64(554),
 'loan': np.int64(1185),
 'pay': np.int64(1554),
 'back': np.int64(138),
 'mid': np.int64(1320),
 'pl': np.int64(1590),
 'lt gt': np.int64(1251),
 'gt dollar': np.int64(870),
 'pay back': np.int64(1555),
 'drive': np.int64(572),
 'rain': np.int64(1688),
 'mrt': np.int64(1391),
 'station': np.int64(1989),
 'lor': np.int64(1207),
 'alreadi': np.int64(57),
 'ask': np.int64(104),
 'go': np.int64(805),
 'ask go': np.int64(107),
 'marriag': np.int64(1285),
 'bite': np.int64(187),
 'bt': np.int64(230),
 'love': np.int64(1224),
 'danc': np.int64(482),
 'amp': np.int64(63),
 'give': np.int64(797),
 'someth': np.int64(1940),
 'drink': np.int64(571),
 'take': np.int64(2055),
 'vomit': np.int64(2307),
 'might': np.int64(1322),
 'drop': np.int64(573),
 'howev': np.int64(982),
 'let': np.int64(1153),
 'know': np.int64(1093),
 'let know': np.int64(1155),
 'much

In [29]:
from sklearn.naive_bayes import MultinomialNB

In [30]:
spam_detection_model_bow = MultinomialNB()
spam_detection_model_bow.fit(X_train_bow, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3851., 606.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.15,-2. ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 2500)","[[ 6., 3., 3., 5.,17.,21., 0., 9., 2., 2.,20., 0., 4., 9., 9., 5., 0., 4.,27., 6., 6.,13., 0., 0., 2., 6., 4., 0., 5., 4.,..., 4.,65., 4., 5., 0., 1.,66., 4.,49., 4., 9., 6.,18.,35., 3., 4., 5., 4.,27., 2., 7., 5., 8., 0., 4., 4.,33., 3., 3., 0.], [ 0., 0., 0., 0., 0., 0.,10., 0., 8., 2.,17.,14., 0., 0., 0., 0., 4., 6., 0., 4., 0., 4.,10., 8., 4., 0., 0., 4., 0., 0.,..., 0.,19., 0., 0., 5., 5., 0., 0., 9., 1., 0., 0., 3., 0., 0., 0., 0., 0., 3., 2., 0., 0.,17., 4., 0., 0., 0., 0., 0., 5.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 2500)","[[ -8.36, -8.92, -8.92, -8.51, -7.41, -7.21,-10.3 , -8. , -9.21, -9.21, -7.26,-10.3 , -8.69, -8. , -8. , -8.51,-10.3 , -8.69, -6.97, -8.36, -8.36, -7.66,-10.3 ,-10.3 , -9.21, -8.36, -8.69,-10.3 , -8.51, -8.69,..., -8.69, -6.11, -8.69, -8.51,-10.3 , -9.61, -6.1 , -8.69, -6.39, -8.69, -8. , -8.36, -7.36, -6.72, -8.92, -8.69, -8.51, -8.69, -6.97, -9.21, -8.22, -8.51, -8.11,-10.3 , -8.69, -8.69, -6.78, -8.92, -8.92,-10.3 ], [ -9.51, -9.51, -9.51, -9.51, -9.51, -9.51, -7.11, -9.51, -7.31, -8.41, -6.62, -6.8 , -9.51, -9.51, -9.51, -9.51, -7.9 , -7.57, -9.51, -7.9 , -9.51, -7.9 , -7.11, -7.31, -7.9 , -9.51, -9.51, -7.9 , -9.51, -9.51,..., -9.51, -6.52, -9.51, -9.51, -7.72, -7.72, -9.51, -9.51, -7.21, -8.82, -9.51, -9.51, -8.13, -9.51, -9.51, -9.51, -9.51, -9.51, -8.13, -8.41, -9.51, -9.51, -6.62, -7.9 , -9.51, -9.51, -9.51, -9.51, -9.51, -7.72]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2500


In [32]:
y_pred = spam_detection_model_bow.predict(X_test_bow)
y_pred

array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..., 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], shape=(1115,))

In [33]:
from sklearn.metrics import accuracy_score,classification_report

In [34]:
accuracy_score(y_test, y_pred)

0.9838565022421525

In [35]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       974
           1       0.96      0.91      0.93       141

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



#### Creating The TF-IDF Model

##### Train Test Split

In [36]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(corpus,y,test_size=0.20)

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf = TfidfVectorizer(max_features=100,ngram_range=(1,2))


In [38]:
X_train_tfidf = tf_idf.fit_transform(X_train).toarray()
X_test_tfidf = tf_idf.transform(X_test).toarray()

In [40]:
X_train_tfidf

array([[0.   , 0.   , 0.   , 0.   , 0.439, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , ..., 0.   , 0.628, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.347, 0.   , 0.   , 0.   , 0.   , 0.   , 0.399, 0.   , 0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.411, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   

In [41]:
X_test_tfidf

array([[0.   , 0.   , 0.76 , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   

In [42]:
X_train_tfidf.shape, X_test_tfidf.shape

((4457, 100), (1115, 100))

In [43]:
tf_idf.vocabulary_

{'sorri': np.int64(71),
 'call': np.int64(4),
 'later': np.int64(33),
 'text': np.int64(76),
 'get': np.int64(19),
 'lt': np.int64(40),
 'gt': np.int64(25),
 'night': np.int64(53),
 'lt gt': np.int64(41),
 'ok': np.int64(56),
 'take': np.int64(74),
 'ur': np.int64(87),
 'time': np.int64(80),
 'repli': np.int64(64),
 'free': np.int64(17),
 'txt': np.int64(85),
 'stop': np.int64(73),
 'come': np.int64(9),
 'number': np.int64(54),
 'ask': np.int64(2),
 'new': np.int64(52),
 'messag': np.int64(44),
 'pleas': np.int64(61),
 'yeah': np.int64(98),
 'alreadi': np.int64(0),
 'go': np.int64(21),
 'ye': np.int64(97),
 'way': np.int64(91),
 'thing': np.int64(78),
 'got': np.int64(23),
 'im': np.int64(31),
 'dont': np.int64(13),
 'know': np.int64(32),
 'want': np.int64(89),
 'say': np.int64(67),
 'good': np.int64(22),
 'win': np.int64(94),
 'see': np.int64(68),
 'co': np.int64(8),
 'uk': np.int64(86),
 'thank': np.int64(77),
 'mobil': np.int64(47),
 'still': np.int64(72),
 'said': np.int64(66),
 'n

In [45]:
from sklearn.naive_bayes import MultinomialNB

spam_detection_model_tfidf = MultinomialNB()
spam_detection_model_tfidf.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3851., 606.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.15,-2. ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 100)","[[ 49.26, 29.39, 53. , 55.09, 99.97, 29.27, 4.46, 0. , 29.2 ,127.61, 73.3 , 86.05, 50.42, 53.03, 38.03, 34.27, 21.34, 25.37, 32.98,136.52, 43.97,170.48, 97.02,101.61, 49.24, 89.44, 39.49, 53.37, 51.08, 76.71,..., 3.23, 73.95, 71.94, 19.97, 63.11, 59.44, 40. , 43.9 , 48.78, 65.11, 91.42, 58.88, 35.27, 0. , 37. , 4.04, 0.41, 83.72, 43.24, 85.02, 51.8 , 47.84, 34.29, 53.81, 8.66, 56.33, 0.37, 45.19, 52.39, 28.87], [ 1. , 0. , 3.18, 9.83,114.1 , 3.05, 34.25, 47.17, 16.29, 1.44, 0.35, 13.89, 7.57, 3.41, 3.83, 0.7 , 14.21, 58.66, 4.51, 23.5 , 3.4 , 10.53, 4.04, 1.22, 4.44, 0. , 0.41, 1.39, 5.96, 0.62,..., 31.44, 1.22, 3.53, 39.37, 5.6 , 4.1 , 44.02, 5.54, 1.26, 4.89, 6.23, 13.58, 2.88, 27.43, 13.86, 58.84, 27.76, 38.56, 7.12, 9.58, 0.36, 0.28, 19.02, 3.07, 28.67, 0.77, 33.23, 7.04, 0.34, 4.41]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 100)","[[-4.65,-5.15,-4.57,-4.54,-3.95,-5.15,-6.87,-8.56,-5.16,-3.71,-4.26,-4.1 ,-4.62,-4.57,-4.9 ,-5. ,-5.46,-5.29,-5.04,-3.64,-4.76,-3.42,-3.98,-3.93,-4.65,-4.06,-4.86,-4.57,-4.61,-4.21,...,-7.12,-4.25,-4.27,-5.52,-4.4 ,-4.46,-4.85,-4.76,-4.66,-4.37,-4.04,-4.47,-4.97,-8.56,-4.93,-6.94,-8.22,-4.12,-4.77,-4.11,-4.6 ,-4.67,-5. ,-4.56,-6.3 ,-4.51,-8.25,-4.73,-4.59,-5.17], [-6.47,-7.16,-5.73,-4.78,-2.42,-5.76,-3.6 ,-3.29,-4.31,-6.27,-6.87,-4.46,-5.02,-5.68,-5.59,-6.63,-4.44,-3.07,-5.46,-3.97,-5.68,-4.72,-5.55,-6.37,-5.47,-7.16,-6.82,-6.29,-5.22,-6.68,...,-3.68,-6.37,-5.65,-3.47,-5.28,-5.53,-3.36,-5.29,-6.35,-5.39,-5.18,-4.48,-5.81,-3.82,-4.46,-3.07,-3.8 ,-3.49,-5.07,-4.8 ,-6.85,-6.91,-4.17,-5.76,-3.77,-6.59,-3.63,-5.08,-6.87,-5.47]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,100


In [46]:
#prediction
y_pred=spam_detection_model_tfidf.predict(X_test_tfidf)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], shape=(1115,))

In [47]:
score=accuracy_score(y_test,y_pred)
print(score)

0.9596412556053812


In [48]:
from sklearn.metrics import classification_report
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.99      0.96      0.98      1001
           1       0.74      0.92      0.82       114

    accuracy                           0.96      1115
   macro avg       0.87      0.94      0.90      1115
weighted avg       0.97      0.96      0.96      1115

